# SD-WAN offline-RL sweep (Colab GPU rig)

Upload `colab/repo.zip` and `colab/datasets.zip` from the project, then run the cells in order. Runtime -> Change runtime type -> GPU (T4).

In [ ]:
!pip install -q "d3rlpy==2.8.0"

In [ ]:
from google.colab import files
import zipfile, os
print("Select repo.zip AND datasets.zip in the file dialog.")
up = files.upload()
for name in up:
    with zipfile.ZipFile(name) as z:
        z.extractall("/content")
    print(f"extracted {name}")
os.makedirs("/content/analysis", exist_ok=True)

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Edit this grid freely (see scripts/offline_sweep.py for options).
!PYTHONPATH=/content python scripts/offline_sweep.py \
    --algos discrete_cql discrete_bc \
    --datasets d1 d2 d3 \
    --steps-list 50000 \
    --alphas 0.5 1.0 4.0 \
    --critics 1 2 \
    --device cuda:0

In [ ]:
from google.colab import files
import zipfile, glob, os
if os.path.exists("/content/analysis/offline_sweep.csv"):
    files.download("/content/analysis/offline_sweep.csv")
with zipfile.ZipFile("/content/models.zip", "w") as z:
    for p in glob.glob("/content/offline/models/*.d3"):
        z.write(p, arcname=os.path.basename(p))
files.download("/content/models.zip")